In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Per-model prompt caching across Gemini 4 Carbon, Neon, and Hydrogen

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fcontext-caching%2Fgemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Authors |
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Gemini 4 Carbon, Neon, and Hydrogen use automatic **Implicit Context Caching** (95% discount on cached input tokens) once a shared prompt prefix reaches **4,096 tokens**. Explicit caching is not supported on Gemini 4.

| Model | Uncached Input (per 1M) | Cached Input (per 1M) | Output + Thinking (per 1M) |
| :--- | :--- | :--- | :--- |
| **Gemini 4 Hydrogen** (`gemini-4-hydrogen`) | \$1.50 | **\$0.075** | \$7.50 |
| **Gemini 4 Neon** (`gemini-4-neon`) | \$3.00 | **\$0.150** | \$15.00 |
| **Gemini 4 Carbon** (`gemini-4-carbon`, standard) | \$4.00 (\$2.00 promo through Nov 30) | **\$0.200** (\$0.100 promo) | \$20.00 (\$10.00 promo) |

Key behaviors covered in this notebook:
- **Exact prefix matching from token 0:** Tool declarations sit ahead of `system_instruction` and `contents`. Prepending a turn counter, dropping a tool, or reordering tools drops cache hits to 0%.
- **Parallel fan-out warmup:** Launching concurrent workers against a cold prefix causes calls 2 through N to miss. Awaiting one warmup call first lets the parallel batch reuse the cache.
- **Per-model KV caches:** Handing off from Carbon or Neon to Hydrogen misses on Hydrogen's first turn (\$1.50/1M), then hits \$0.075/1M on consecutive Hydrogen turns while the planner model's own cache stays warm.
- **CI checks and retention controls:** Catch prefix regressions locally with a SHA-256 check, and inspect project `DURABLE` vs. `EPHEMERAL` cache retention for zero data retention (ZDR).

## Get started

### Install Google Gen AI SDK

> **Note:** You can safely ignore `pip` dependency resolver errors from unrelated pre-installed packages in Colab.

In [ ]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment

Authenticate your account if running in Google Colab.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [3]:
import hashlib
import json
import os
import time
import urllib.error
import urllib.request
import uuid
from concurrent.futures import ThreadPoolExecutor
from typing import Any

import google.auth
from google import genai
from google.auth.transport.requests import Request
from google.genai import types
from google.genai.errors import APIError

### Set Google Cloud project information

Set your Google Cloud project ID and initialize `genai.Client`.

In [4]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "global"  # @param ["global", "us", "eu"]

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        timeout=240000,
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
)

### Model rates and shared prompt prefix

Set up a cost helper and a static prompt prefix above the 4,096-token caching threshold.

In [5]:
CARBON_MODEL = "gemini-4-carbon"
NEON_MODEL = "gemini-4-neon"
HYDROGEN_MODEL = "gemini-4-hydrogen"

# Standard per-1M input token pricing: (uncached_usd, cached_usd)
INPUT_RATES_PER_1M: dict[str, tuple[float, float]] = {
    HYDROGEN_MODEL: (1.50, 0.075),
    NEON_MODEL: (3.00, 0.150),
    CARBON_MODEL: (4.00, 0.200),
}


def input_cost_per_1k_calls(
    model: str, prompt_tokens: int, cached_tokens: int
) -> float:
    uncached_rate, cached_rate = INPUT_RATES_PER_1M[model]
    uncached = max(0, prompt_tokens - cached_tokens)
    cost_one_call = (
        uncached * uncached_rate + cached_tokens * cached_rate
    ) / 1_000_000
    return cost_one_call * 1_000


RUN_TAG = uuid.uuid4().hex[:8]

SHARED_REPO_PREFIX = (
    f"You are a distributed systems coding agent (run {RUN_TAG}). "
    "Answer concisely in 1 sentence.\n\n"
    + "\n".join(
        f"## Service Module svc-{i:03d}\n"
        f"- Owner: platform-core | Region: us-east1 | Protocol: gRPC + Protobuf\n"
        f"- Retry Policy: exponential_jitter(base_ms=80, cap_ms={600 + i})\n"
        f"- Idempotency: required header `X-Idempotency-Key` with 24h Redis TTL."
        for i in range(1, 185)
    )
)

## Explicit caching check

If you are migrating from Gemini 3.x, remove `caches.create` calls and `cached_content` fields. Gemini 4 rejects explicit cache creation with `HTTP 400`.

In [6]:
try:
    client.caches.create(
        model=NEON_MODEL,
        config=types.CreateCachedContentConfig(
            contents=[SHARED_REPO_PREFIX],
            ttl="300s",
        ),
    )
except APIError as err:
    print(f"Expected guardrail -> HTTP {err.code}: {err.message[:85]}")

Expected guardrail -> HTTP 400: Model `gemini-4-neon` does not support explicit context caching.


## Prefix ordering: dynamic-first vs. static-first

Implicit caching matches tokens from the start of the request (`tools` + `system_instruction` + `contents`). Putting changing text at the start of `system_instruction` results in 0 cached tokens.

### Dynamic header at the start (misses cache)

In [7]:
anti_pattern_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "anti-pattern demo"
)

for turn in (1, 2):
    # Bad: dynamic header at index 0 changes every turn and breaks prefix matching
    bad_sys = (
        f"[Turn={turn} | RequestTimestamp={1763500000 + turn}]\n"
        + anti_pattern_prefix
    )
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"What is cap_ms for svc-0{turn}0?",
        config=types.GenerateContentConfig(
            system_instruction=bad_sys,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    u = resp.usage_metadata
    print(
        f"Dynamic-first Turn {turn}: prompt={u.prompt_token_count}, "
        f"cached={u.cached_content_token_count or 0}"
    )

Dynamic-first Turn 1: prompt=14813, cached=0
Dynamic-first Turn 2: prompt=14813, cached=0


### Static instructions first, dynamic text at the end (hits cache)

In [8]:
static_first_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "static-first demo"
)

for turn in (1, 2, 3):
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"[Turn={turn}] What is cap_ms for svc-0{turn}0?",
        config=types.GenerateContentConfig(
            system_instruction=static_first_prefix,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    u = resp.usage_metadata
    cached = u.cached_content_token_count or 0
    hit_pct = 100.0 * cached / u.prompt_token_count
    print(
        f"Static-first Turn {turn}: prompt={u.prompt_token_count}, "
        f"cached={cached} ({hit_pct:4.1f}%)"
    )
    time.sleep(1.0)

Static-first Turn 1: prompt=14798, cached=0 ( 0.0%)
Static-first Turn 2: prompt=14798, cached=12744 (86.1%)
Static-first Turn 3: prompt=14798, cached=12749 (86.2%)


## Tool definitions and prefix matching

Tool declarations sit ahead of `system_instruction` in the cached prefix. Sending a smaller tool subset or reordering the same tools starts a cold cache even when `system_instruction` is unchanged.

For subagent fan-out, pass the same sorted tool list and enforce per-subagent permissions in your harness, or expect one cold warmup call per tool profile.

In [9]:
tool_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "tool-order cache demo"
)

tool_a = types.FunctionDeclaration(
    name="read_file",
    description="Read a repository file by path.",
    parameters_json_schema={
        "type": "object",
        "properties": {"path": {"type": "string"}},
        "required": ["path"],
    },
)
tool_b = types.FunctionDeclaration(
    name="run_tests",
    description="Run unit tests for a service module.",
    parameters_json_schema={
        "type": "object",
        "properties": {"module": {"type": "string"}},
        "required": ["module"],
    },
)
tool_c = types.FunctionDeclaration(
    name="search_code",
    description="Search symbols across the repository.",
    parameters_json_schema={
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
)

all_tools = [types.Tool(function_declarations=[tool_a, tool_b, tool_c])]
sub_tools = [types.Tool(function_declarations=[tool_a])]
rev_tools = [types.Tool(function_declarations=[tool_c, tool_b, tool_a])]

tool_cases = [
    ("1. Warmup   [A,B,C]", all_tools),
    ("2. Same     [A,B,C]", all_tools),
    ("3. Subset   [A]    ", sub_tools),
    ("4. Subset   [A] #2 ", sub_tools),
    ("5. Reorder  [C,B,A]", rev_tools),
    ("6. Back     [A,B,C]", all_tools),
]

for idx, (label, tools_cfg) in enumerate(tool_cases, start=1):
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"Reply OK {idx} without calling any tools.",
        config=types.GenerateContentConfig(
            system_instruction=tool_prefix,
            tools=tools_cfg,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    u = resp.usage_metadata
    p_tok = u.prompt_token_count or 0
    c_tok = u.cached_content_token_count or 0
    pct = (100.0 * c_tok / p_tok) if p_tok else 0.0
    print(f"{label} | prompt={p_tok:>5} | cached={c_tok:>5} ({pct:4.1f}%)")
    time.sleep(1.0)

1. Warmup   [A,B,C] | prompt=14935 | cached=    0 ( 0.0%)
2. Same     [A,B,C] | prompt=14935 | cached=12881 (86.2%)
3. Subset   [A]     | prompt=14839 | cached=    0 ( 0.0%)
4. Subset   [A] #2  | prompt=14839 | cached=12785 (86.2%)
5. Reorder  [C,B,A] | prompt=14935 | cached=    0 ( 0.0%)
6. Back     [A,B,C] | prompt=14935 | cached=12886 (86.3%)


## Parallel fan-out: cold start vs. pre-warmed

Launching concurrent subagents against a cold prefix causes calls 2 through N to miss before the first prefill lands. Awaiting worker 1 first, then running workers 2 through N in parallel, lets the batch reuse the warm prefix across the same total number of tasks.

In [10]:
def run_worker(prefix_text: str, worker_id: int) -> tuple[int, int, int]:
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"Worker {worker_id}: report cap_ms for svc-0{worker_id}0.",
        config=types.GenerateContentConfig(
            system_instruction=prefix_text,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    u = resp.usage_metadata
    return worker_id, u.prompt_token_count or 0, u.cached_content_token_count or 0


# A. Cold fan-out: all 5 workers start simultaneously on an unseen prefix
cold_prefix = SHARED_REPO_PREFIX.replace("distributed systems", "cold fan-out demo")
with ThreadPoolExecutor(max_workers=5) as pool:
    cold_results = list(pool.map(lambda w: run_worker(cold_prefix, w), range(1, 6)))

cold_cost = sum(
    input_cost_per_1k_calls(HYDROGEN_MODEL, p, c) for _, p, c in cold_results
)
cold_hits = sum(1 for _, _, c in cold_results if c > 0)
print(
    f"Cold fan-out (5 parallel):      {cold_hits}/5 hits | "
    f"${cold_cost:6.2f} / 1K batches"
)

# B. Pre-warmed fan-out: await worker 1 first, then launch workers 2-5 in parallel
warm_prefix = SHARED_REPO_PREFIX.replace("distributed systems", "warm fan-out demo")
first_result = run_worker(warm_prefix, 1)
time.sleep(2.0)

with ThreadPoolExecutor(max_workers=4) as pool:
    rest_results = list(pool.map(lambda w: run_worker(warm_prefix, w), range(2, 6)))

warm_results = [first_result, *rest_results]
warm_cost = sum(
    input_cost_per_1k_calls(HYDROGEN_MODEL, p, c) for _, p, c in warm_results
)
warm_hits = sum(1 for _, _, c in rest_results if c > 0)
print(
    f"Pre-warmed (1 first + 4 par):   {warm_hits}/4 hits | "
    f"${warm_cost:6.2f} / 1K batches"
)

Cold fan-out (5 parallel):      0/5 hits | $110.98 / 1K batches
Pre-warmed (1 first + 4 par):   4/4 hits | $ 38.34 / 1K batches


## Per-model KV cache isolation across Carbon, Neon, and Hydrogen

Run a 6-step sequence over the same shared prefix (`Carbon -> Neon -> Hydrogen -> Hydrogen -> Neon -> Carbon`):
- Steps 1 to 3 each use a model for the first time, so `cached` is `0` on all three.
- Step 4 stays on Hydrogen and hits Hydrogen's cache.
- Steps 5 and 6 switch back to Neon and Carbon and hit each model's still-warm cache.

In [11]:
isolation_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "cross-model isolation demo"
)

handoff_schedule = [
    (CARBON_MODEL, "Step 1 (Carbon cold prefill): check cap_ms for svc-005."),
    (NEON_MODEL, "Step 2 (Neon cold prefill): check cap_ms for svc-015."),
    (HYDROGEN_MODEL, "Step 3 (Hydrogen cold prefill): check cap_ms for svc-025."),
    (HYDROGEN_MODEL, "Step 4 (Hydrogen consecutive): check cap_ms for svc-035."),
    (NEON_MODEL, "Step 5 (Hand back to warm Neon): check cap_ms for svc-045."),
    (CARBON_MODEL, "Step 6 (Hand back to warm Carbon): check cap_ms for svc-055."),
]

history: list[types.Content] = []
print(
    f"{'Step & Model':<28} | {'Prompt':>7} | {'Cached':>7} | "
    f"{'Hit %':>6} | {'$/1K Calls':>10}"
)
print("-" * 70)

for idx, (model_id, user_msg) in enumerate(handoff_schedule, start=1):
    history.append(
        types.Content(role="user", parts=[types.Part.from_text(text=user_msg)])
    )
    resp = client.models.generate_content(
        model=model_id,
        contents=history,
        config=types.GenerateContentConfig(
            system_instruction=isolation_prefix,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    history.append(resp.candidates[0].content)
    u = resp.usage_metadata
    prompt_tok = u.prompt_token_count or 0
    cached_tok = u.cached_content_token_count or 0
    hit_pct = (100.0 * cached_tok / prompt_tok) if prompt_tok else 0.0
    cost_1k = input_cost_per_1k_calls(model_id, prompt_tok, cached_tok)
    label = f"{idx}. {model_id}"
    print(
        f"{label:<28} | {prompt_tok:>7} | {cached_tok:>7} | "
        f"{hit_pct:>5.1f}% | ${cost_1k:>9.3f}"
    )

Step & Model                 |  Prompt |  Cached |  Hit % | $/1K Calls
----------------------------------------------------------------------
1. gemini-4-carbon           |   14802 |       0 |   0.0% | $   59.208
2. gemini-4-neon             |   14860 |       0 |   0.0% | $   44.580
3. gemini-4-hydrogen         |   14904 |       0 |   0.0% | $   22.356
4. gemini-4-hydrogen         |   14953 |   12855 |  86.0% | $    4.111
5. gemini-4-neon             |   15016 |   12811 |  85.3% | $    8.537
6. gemini-4-carbon           |   15069 |   12748 |  84.6% | $   11.834


## 12-turn agent cost comparison

Compare a 12-turn coding loop (1 plan turn, 10 worker turns, 1 review turn; 15K starting input tokens, +500 input and 1,200 thinking/output tokens per turn) across five setups.

Each model only caches tokens up to the prefix length it saw on its previous turn. When Turn 12 hands back from Hydrogen to Neon, Neon can reuse Turn 1's 15K prefix (setup D) or pay full input price if evicted (setup E).

In [12]:
OUTPUT_RATES_PER_1M: dict[str, float] = {
    HYDROGEN_MODEL: 7.50,
    NEON_MODEL: 15.00,
    CARBON_MODEL: 20.00,
}


def simulate_12_turn_tco(
    schedule: list[tuple[str, bool]],
    base_in: int = 15_000,
    turn_growth: int = 500,
    out_per_turn: int = 1_200,
) -> float:
    warmed_prefix: dict[str, int] = {}
    total_per_1k = 0.0
    for idx, (mdl, use_cache) in enumerate(schedule):
        in_tok = base_in + idx * turn_growth
        cached_tok = warmed_prefix.get(mdl, 0) if use_cache else 0
        if use_cache:
            warmed_prefix[mdl] = in_tok
        in_cost = input_cost_per_1k_calls(mdl, in_tok, cached_tok)
        out_cost = (out_per_turn * OUTPUT_RATES_PER_1M[mdl] / 1_000_000) * 1_000
        total_per_1k += in_cost + out_cost
    return total_per_1k


schedules = {
    "A. All Carbon (broken prefix, 0% cache)": [(CARBON_MODEL, False)] * 12,
    "B. All Carbon (stable prefix, 95% cache)": [(CARBON_MODEL, True)] * 12,
    "C. All Neon (stable prefix, 95% cache)": [(NEON_MODEL, True)] * 12,
    "D. Neon plan/review + 10 Hydrogen (warm 15K)": (
        [(NEON_MODEL, True)] + [(HYDROGEN_MODEL, True)] * 10 + [(NEON_MODEL, True)]
    ),
    "E. Neon plan/review + 10 Hydrogen (cold review)": (
        [(NEON_MODEL, True)] + [(HYDROGEN_MODEL, True)] * 10 + [(NEON_MODEL, False)]
    ),
}

for label, sched in schedules.items():
    tco = simulate_12_turn_tco(sched)
    print(f"{label:<47} -> ${tco:>7.2f} / 1,000 agent runs")

A. All Carbon (broken prefix, 0% cache)         -> $1140.00 / 1,000 agent runs
B. All Carbon (stable prefix, 95% cache)        -> $ 408.50 / 1,000 agent runs
C. All Neon (stable prefix, 95% cache)          -> $ 306.38 / 1,000 agent runs
D. Neon plan/review + 10 Hydrogen (warm 15K)    -> $ 231.56 / 1,000 agent runs
E. Neon plan/review + 10 Hydrogen (cold review) -> $ 274.31 / 1,000 agent runs


## Batched handoffs on the Interactions API

Chaining turns with `previous_interaction_id` (`store=True`) follows the same per-model cache behavior. Keeping `system_instruction` fixed hits `interaction.usage.total_cached_tokens` on consecutive `gemini-4-hydrogen` worker turns and reuses Neon's Turn 1 prefix when returning to Neon on Turn 5 (implicit caching is best-effort across serving replicas, so single runs can occasionally miss).

In [13]:
interactions_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "interactions-api cache demo"
)
stored_ids: list[str] = []

turns_spec = [
    (NEON_MODEL, "low", "Plan: list svc-001 and svc-002 cap_ms values."),
    (HYDROGEN_MODEL, "low", "Worker 1: compute their sum."),
    (HYDROGEN_MODEL, "low", "Worker 2: compute their difference."),
    (HYDROGEN_MODEL, "low", "Worker 3: format both numbers as JSON."),
    (NEON_MODEL, "low", "Verify: confirm the JSON matches svc-001 and svc-002."),
]

prev_id: str | None = None
print(f"{'Turn & Model':<28} | {'Input Tok':>9} | {'Cached Tok':>10} | {'Hit %':>6}")
print("-" * 62)

for idx, (mdl, level, prompt) in enumerate(turns_spec, start=1):
    kwargs: dict[str, Any] = {
        "model": mdl,
        "system_instruction": interactions_prefix,
        "input": prompt,
        "generation_config": {"thinking_level": level},
        "store": True,
        "timeout": 180,
    }
    if prev_id is not None:
        kwargs["previous_interaction_id"] = prev_id

    inter = client.interactions.create(**kwargs)
    stored_ids.append(inter.id)
    prev_id = inter.id

    in_tok = inter.usage.total_input_tokens or 0
    cached_tok = inter.usage.total_cached_tokens or 0
    pct = (100.0 * cached_tok / in_tok) if in_tok else 0.0
    print(f"{f'{idx}. {mdl}':<28} | {in_tok:>9} | {cached_tok:>10} | {pct:>5.1f}%")

Turn & Model                 | Input Tok | Cached Tok |  Hit %
--------------------------------------------------------------
1. gemini-4-neon             |     14804 |          0 |   0.0%
2. gemini-4-hydrogen         |     14852 |          0 |   0.0%
3. gemini-4-hydrogen         |     14965 |      12803 |  85.6%
4. gemini-4-hydrogen         |     15107 |      12916 |  85.5%
5. gemini-4-neon             |     15330 |      12747 |  83.2%


## Testing prefix stability in CI

Because broken prefixes return `200 OK` without raising an error, add a local unit test that builds your request payload for two simulated turns (different turn counter, timestamp, and user query) and checks that the serialized tools and static prefix have identical SHA-256 hashes.

If you build agents with Google ADK, put long-lived system headers in `static_instruction` and per-turn steering in `instruction` so the cache boundary stays explicit.

In [14]:
def prefix_sha256(
    tool_names: list[str], system_instruction: str, static_blocks: list[str]
) -> str:
    payload = json.dumps(
        {
            "tools": tool_names,
            "system_instruction": system_instruction,
            "static_blocks": static_blocks,
        },
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]


def check_prefix_stability(
    name: str, turn_1_args: tuple, turn_2_args: tuple
) -> None:
    h1 = prefix_sha256(*turn_1_args)
    h2 = prefix_sha256(*turn_2_args)
    status = "PASS (byte-identical)" if h1 == h2 else "FAIL (cache broken)"
    print(f"{name:<32} -> {status} [{h1} vs {h2}]")


base_tools = ["read_file", "run_tests", "search_code"]
check_prefix_stability(
    "Dynamic turn header in system",
    (base_tools, "[Turn=1]\n" + SHARED_REPO_PREFIX[:200], []),
    (base_tools, "[Turn=2]\n" + SHARED_REPO_PREFIX[:200], []),
)
check_prefix_stability(
    "Reordered tool declarations",
    (base_tools, SHARED_REPO_PREFIX[:200], []),
    (list(reversed(base_tools)), SHARED_REPO_PREFIX[:200], []),
)
check_prefix_stability(
    "Sorted tools + static prefix",
    (sorted(base_tools), SHARED_REPO_PREFIX[:200], ["repo-schema-v1"]),
    (sorted(base_tools), SHARED_REPO_PREFIX[:200], ["repo-schema-v1"]),
)

Dynamic turn header in system    -> FAIL (cache broken) [648481e36ad5e45a vs 53baf524f27e67a8]
Reordered tool declarations      -> FAIL (cache broken) [0b68a55f516bb567 vs f77e4a1d9c21b77f]
Sorted tools + static prefix     -> PASS (byte-identical) [94becbf8b403c8d1 vs 94becbf8b403c8d1]


## Zero data retention and cache retention modes

Implicit caching uses two project-level storage tiers on Google Cloud:
- **`DURABLE` (default):** Prefixes stay in memory and spill to project-isolated, Google-encrypted storage (deleted within 24 hours), improving hit rates across longer idle gaps.
- **`EPHEMERAL`:** Prefixes stay in RAM only (nothing written at rest), with no guaranteed minimum TTL. It works best for tight back-to-back agent loops rather than 20-minute idle gaps.

For strict [Zero Data Retention (ZDR)](https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/zero-data-retention#controlling-data-caching), set `retentionType` to `EPHEMERAL` on your project's `cacheConfig` and pass `store=False` on Interactions API calls (`store` defaults to `True`).

> **Note:** Starting January 12, 2027, Gemini 4 Hydrogen and later models add TTL-configurable cache control with `1.25x` default cache write pricing. See the [Context caching overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/context-cache/context-cache-overview) for details.

In [15]:
# Read-only check of project cacheConfig (requires roles/aiplatform.viewer)
creds, _ = google.auth.default()
creds.refresh(Request())

config_url = (
    f"https://us-central1-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/cacheConfig"
)
req = urllib.request.Request(
    config_url, headers={"Authorization": f"Bearer {creds.token}"}
)
try:
    with urllib.request.urlopen(req, timeout=20) as resp:
        cache_cfg = json.loads(resp.read().decode("utf-8"))
    retention = cache_cfg.get("retentionConfig", {}).get("retentionType", "DURABLE")
    print(
        f"Project cacheConfig -> disableCache={cache_cfg.get('disableCache', False)}, "
        f"retentionType={retention}"
    )
except urllib.error.HTTPError as err:
    print(f"Could not read cacheConfig (HTTP {err.code}); verify IAM viewer role.")

# To switch to RAM-only caching (requires roles/aiplatform.admin), PATCH config_url:
# {"name": f"projects/{PROJECT_ID}/cacheConfig", "disableCache": False,
#  "retentionConfig": {"retentionType": "EPHEMERAL"}}

Project cacheConfig -> disableCache=False, retentionType=DURABLE


## Cleaning up

Delete the stored interactions from the previous section (stored interactions otherwise expire after 7 days).

In [16]:
for interaction_id in stored_ids:
    client.interactions.delete(interaction_id)

print(f"Deleted {len(stored_ids)} stored interactions.")

Deleted 5 stored interactions.
